# Cylindrical equilibrium $\nabla\cdot T + f = 0$ (vibe 000073)

Reproducing the classic hand derivation of the balance equations for a
continuous medium in cylindrical coordinates.  We take the stress balance
$\nabla\cdot T + f = 0$, represent the rank-2 stress $T$ in the cylindrical
physical frame $\mathbf e_r,\mathbf e_\theta,\mathbf e_z$, and let tender
compute the divergence — differentiating the components **and** the moving
basis vectors together (via the frame connection
$\partial_\theta\mathbf e_r=\mathbf e_\theta$,
$\partial_\theta\mathbf e_\theta=-\mathbf e_r$).  The three scalar balance
equations fall out and match the standard textbook form.

In [ ]:
import tender as t
import tender.basis as tb
import tender.derivation as td
from IPython.display import Math, display

ws = t.Workspace()


def disp(*exprs, lhs=None):
    for e in exprs:
        s = e.latex() if hasattr(e, "latex") else str(e)
        display(Math((lhs + " = " if lhs else "") + s))

## 0. The cylindrical chart

The mapping $x=r\cos\theta,\ y=r\sin\theta,\ z=z$ targets the orthonormal
Cartesian world frame.  `nonneg=("r",)` licenses $\sqrt{r^2}=r$.

In [ ]:
WCS = ws.wcs()
r, th, z = ws.coords("r", r"\theta", "z", nonneg=("r",))
cyl = ws.chart(WCS, [r, th, z], [r * t.cos(th), r * t.sin(th), z])
names = ["r", r"\theta", "z"]

## 1. The physical orthonormal frame

The frame $\mathbf e_i=\mathbf g_i/h_i$ is derived from the mapping — no
hand-supplied scale factors.

In [ ]:
frame = cyl.physical_frame()
e = [frame.direction(k) for k in range(3)]
disp(*e)
print("orthonormal:", frame.is_orthonormal)

## 2. How the frame turns — the connection $\partial_j\mathbf e_i$

Only the $\theta$-derivatives are nonzero:
$\partial_\theta\mathbf e_r=\mathbf e_\theta$,
$\partial_\theta\mathbf e_\theta=-\mathbf e_r$.  These are what make the
curvilinear divergence differ from the flat one; tender derives them, they are
not tabulated.

In [ ]:
def field_latex(components):
    parts = []
    for c, nm in zip(components, names):
        s = td.simplify_scalars(c).latex()
        if s == "0":
            continue
        vec = rf"\mathbf{{e}}_{{{nm}}}"
        parts.append(vec if s == "1" else rf"\left({s}\right)\,{vec}")
    return " + ".join(parts).replace("+ -", "-") or "0"


for i, j, lbl in [(0, 1, r"\partial_\theta \mathbf e_r"),
                  (1, 1, r"\partial_\theta \mathbf e_\theta"),
                  (0, 0, r"\partial_r \mathbf e_r")]:
    display(Math(lbl + " = " + field_latex(cyl.connection_coefficients(i, j))))

## 3. The stress field, represented in the frame  (eq 4)

$T$ is an abstract rank-2 field; `expand_in_basis` writes it as
$\sum_{ij} T_{ij}\,\mathbf e_i\otimes\mathbf e_j$, minting the physical
components $T_{ij}$ as **fields** of the coordinates — so the divergence can
differentiate them (vibe 000073).

In [ ]:
T = ws.field("T", 2)  # depends on r, theta, z
T_cyl = td.canonicalize(td.unroll_sums(
    tb.expand_in_basis(T, frame, tb.Variance.Covariant)))
disp(T_cyl, lhs="T")

## 4. The divergence $\nabla\cdot T$, on the frame  (eq 7)

`cyl.div` applies $\nabla\cdot=\sum_i\frac1{h_i}\mathbf e_i\cdot\partial_i$,
Leibniz-differentiating components and basis vectors.  We project the result
onto each $\mathbf e_i$ to read the three scalar components.

In [ ]:
def reduce_scalar(x):
    x = tb.simplify_basis_dot(td.expand_products(x), frame)
    return td.simplify_scalars(td.fold_arithmetic(
        td.eval_delta_concrete(td.canonicalize(x))))


def comp(vec, i):
    return reduce_scalar(vec @ e[i])


def Tij(i, j):
    return reduce_scalar(e[i] @ T_cyl @ e[j])


div_T = cyl.div(T_cyl)
for i, nm in enumerate(names):
    display(Math(rf"(\nabla\cdot T)_{{{nm}}} = " + comp(div_T, i).latex()))

## 5. Cross-check against the textbook equilibrium equations

The standard cylindrical divergence of a rank-2 field (contracting the first
index):

$$(\nabla\cdot T)_r = \partial_r T_{rr} + \tfrac1r\partial_\theta T_{\theta r}
 + \partial_z T_{zr} + \tfrac{T_{rr}-T_{\theta\theta}}{r}$$
$$(\nabla\cdot T)_\theta = \partial_r T_{r\theta}
 + \tfrac1r\partial_\theta T_{\theta\theta} + \partial_z T_{z\theta}
 + \tfrac{T_{r\theta}+T_{\theta r}}{r}$$
$$(\nabla\cdot T)_z = \partial_r T_{rz}
 + \tfrac1r\partial_\theta T_{\theta z} + \partial_z T_{zz} + \tfrac{T_{rz}}{r}$$

For a **symmetric** stress $T_{\theta r}=T_{r\theta}$, the $\theta$-component
carries the classic $2T_{r\theta}/r$ shear term.

In [ ]:
def d(x, coord):
    return td.simplify_scalars(td.partial(x, coord))


def same(a, b):
    return td.simplify_scalars(a - b).latex() == "0"


div_r, div_th, div_z = (comp(div_T, i) for i in range(3))
expect_r = d(Tij(0,0),r) + d(Tij(1,0),th)/r + d(Tij(2,0),z) + (Tij(0,0)-Tij(1,1))/r
expect_th = d(Tij(0,1),r) + d(Tij(1,1),th)/r + d(Tij(2,1),z) + (Tij(0,1)+Tij(1,0))/r
expect_z = d(Tij(0,2),r) + d(Tij(1,2),th)/r + d(Tij(2,2),z) + Tij(0,2)/r

assert same(div_r, expect_r)
assert same(div_th, expect_th)
assert same(div_z, expect_z)
print("div T matches the standard cylindrical equilibrium equations")

## 6. The boiler formula  (eq 9)

Assume $T$ and $f$ depend on $r$ only and there is no shear.  The $\theta$- and
$z$-equations vanish and the $r$-equation is the boiler formula
$\partial_r T_{rr} + \tfrac{T_{rr}-T_{\theta\theta}}{r} + f_r = 0$.  It gives
the hoop-stress estimate $T_{\theta\theta}\approx Rp/d$ for a thin pipe.

In [ ]:
boiler = td.simplify_scalars(d(Tij(0,0), r) + (Tij(0,0) - Tij(1,1)) / r)
disp(boiler, lhs=r"\partial_r T_{rr} + \tfrac{T_{rr}-T_{\theta\theta}}{r}")